# Chapter 7 — Source to IR

Step 6 builds **phase B's front half**: the fused typing+lowering pass. From
a Handle's decoration-time snapshot to verified, typed core IR — in one
forward walk, with the *language itself living outside the kernel* as
`lower_ast` rules.

| Piece | Counted lines / cap | What |
|---|---|---|
| `kernel/lower.py` | 94 / 170 | the DRIVER: coherence, rebased locs, name fates, inlining, Derived build-rule dispatch |
| `stdlib/base_lang.py` (satellite) | ~80 | the base dialect's rule pack — the accepted Python subset, as a dict |
| `combinators.build_pipe` (satellite) | ~25 | the `Derived("pipe")` fusion build rule |

Two decisions live in this chapter: **the base dialect is strict** (no
auto-cast insertion — `float(i)` is the user's job, per the ch05 settlement;
a friendlier dialect may choose otherwise in ITS pack), and **the language
is a registration** (widening Python support = adding a dict entry in a
satellite, never editing the driver — numba's `typeinfer.py` failure mode,
structurally excluded).

Glossary terms settled: *name fates, lower_ast rule, build rule,
NoSourceError/StaleSourceError.*

In [1]:
from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.api import jit
from pdum.dsl.kernel.lower import lower_handle
from pdum.dsl.kernel.ops import CORE_OPS
from pdum.dsl.kernel.printer import print_program
from pdum.dsl.stdlib.base_lang import LOWER_RULES


def dist2(cx, cy):
    @jit()
    def go(px, py):
        dx = px - cx
        dy = py - cy
        return dx * dx + dy * dy

    return go


h = dist2(320.0, 240.0)
region = lower_handle(h, LOWER_RULES, CORE_OPS, arg_types=(T.f64, T.f64))
print(print_program(region, name="dist2"))

dist2(%p0: f64, %p1: f64) {
  %0 = core.env {slot = (0,)} : f64
  %1 = core.sub %p0, %0 : f64
  %2 = core.mul %1, %1 : f64
  %3 = core.env {slot = (1,)} : f64
  %4 = core.sub %p1, %3 : f64
  %5 = core.mul %4, %4 : f64
  %6 = core.add %2, %5 : f64
  core.yield %6
}


Source became typed IR: parameters are `%p`s, captures are `core.env` with
**path** attrs (`slot = (0,)` — paths, not flat ints, because inlining nests
environments; marshaling flattens the tree in ch08), and every node carries
an absolute source location, rebased through the snapshot.

In [2]:
from pdum.dsl.kernel.ir import format_loc

sub = next(n for n in region.body[-1].args[0].args[0].args for n in [n] if n.op == "core.sub")
print("this core.sub came from:", format_loc(sub.loc))


# THE STRICT EXPERIENCE, end to end from real source — the base dialect does
# not auto-cast; the error is a starting region with your file and lines:
@jit()
def mixed(x):
    return x + 1  # an i64 literal against an f64 parameter


try:
    lower_handle(mixed, LOWER_RULES, CORE_OPS, arg_types=(T.f64,))
except TypeError as terr:
    print("\nrefused:", terr)


@jit()
def fixed(x):
    return x + float(1)  # the Julia way — and the cast is IN the IR, IN the hash


print()
print(print_program(lower_handle(fixed, LOWER_RULES, CORE_OPS, arg_types=(T.f64,)), name="fixed"))

this core.sub came from: /var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/1762948926.py:12:9

refused: core arithmetic is strict: f64 vs i64 — insert an explicit core.cast [/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/2955962400.py:11:11; /var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/2955962400.py:11:15]

fixed(%p0: f64) {
  %0 = core.const {value = 1} : i64
  %1 = core.cast %0 {to = f64} : f64
  %2 = core.add %p0, %1 : f64
  core.yield %2
}


In [3]:
# Name fates are a CLOSED taxonomy: param, local, capture — anything else is
# loud. Globals have no sanctioned fate yet (no silent freezing — the M0-era
# hazard the guards exist for).
from pdum.dsl.kernel.lower import MissingRule, NameFateError

SOME_GLOBAL = 42


@jit()
def leaky(x):
    return x + SOME_GLOBAL


try:
    lower_handle(leaky, LOWER_RULES, CORE_OPS, arg_types=(T.f64,))
except NameFateError as e:
    print("fate error: ", e)


@jit()
def tup(x):
    return (x, x)


try:
    lower_handle(tup, LOWER_RULES, CORE_OPS, arg_types=(T.f64,))
except MissingRule as e:
    print("missing rule:", e, " <- widening = a dict entry in a rule pack")

fate error:  'SOME_GLOBAL' is neither a parameter, a local, nor a capture [/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/1589131086.py:11:15]; globals have no sanctioned fate yet — capture it


In [4]:
# Inlining: a captured kernel called in the body is fused in — multi-statement
# bodies included (M0 demanded single-return; that restriction is gone).
def make_inner(k):
    @jit()
    def inner(y):
        t = y * k
        return t + t

    return inner


def make_outer(inner, c):
    @jit()
    def outer(x):
        return inner(x) + c

    return outer


fused = lower_handle(make_outer(make_inner(2.0), 10.0), LOWER_RULES, CORE_OPS, arg_types=(T.f64,))
print(print_program(fused, name="fused"))
print()
mul_node = next(n for n in fused.body[-1].args[0].args[0].args if n.op == "core.mul")
print("inlined provenance:", format_loc(mul_node.loc))

fused(%p0: f64) {
  %0 = core.env {slot = (1, 0)} : f64
  %1 = core.mul %p0, %0 : f64
  %2 = core.add %1, %1 : f64
  %3 = core.env {slot = (0,)} : f64
  %4 = core.add %2, %3 : f64
  core.yield %4
}

inlined provenance: /var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/3625836531.py:6:8 (inlined from /var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/3625836531.py:15:11)


Read the env paths: `c` is the outer's capture `(0,)`; the inner kernel sits
at slot 1, so *its* capture `k` is `(1, 0)` — the environment TREE from ch02,
now syntactic in the IR, ready for `EnvLeaf` flattening at ch08. And the
provenance chains: the `core.mul` knows it lives in `inner`'s source,
*inlined from* the call site in `outer`.

In [5]:
# ch04's promise, kept: the pipe build rule lowers Derived("pipe") WITHOUT
# source — pure structure, each stage inlined in sequence. Fusion is now a
# fact at the IR level (execution needs ch09's backend).
from pdum.dsl.combinators import PIPE_BUILDERS, op, register_composition, register_role

register_role("device")  # stand-in, as in ch04: the stdlib package owns this
register_composition("pipe", "device", "device", "fuse")


@op
def padd(k):
    @jit()
    def go(x):
        return x + k

    return go


@op
def pmul(k):
    @jit()
    def go(x):
        return x * k

    return go


pipeline = padd(1.0) | pmul(2.0)
body = lower_handle(pipeline, LOWER_RULES, CORE_OPS, arg_types=(T.f64,), derived=PIPE_BUILDERS)
print(print_program(body, name="fused_pipe"))
print()
print("stage provenance:", format_loc(body.body[-1].args[0].loc))

fused_pipe(%p0: f64) {
  %0 = core.env {slot = (0, 0)} : f64
  %1 = core.add %p0, %0 : f64
  %2 = core.env {slot = (1, 0)} : f64
  %3 = core.mul %1, %2 : f64
  core.yield %3
}

stage provenance: /var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/1837076449.py:23:11 (inlined from <pipeline>:2)


In [6]:
from pdum.dsl import viz

viz.install()
body  # the fused pipeline, rendered — hover %refs to see <pipeline> provenance

Region(params=(Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 0),), loc=None),), body=(Node(op='core.yield', type=f64, args=(Node(op='core.mul', type=f64, args=(Node(op='core.add', type=f64, args=(Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 0),), loc=None), Node(op='core.env', type=f64, args=(), regions=(), attrs=(('slot', (0, 0)),), loc=CallLoc(callee=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/1837076449.py', line=14, col=15), caller=Loc(file='<pipeline>', line=1, col=0)))), regions=(), attrs=(), loc=CallLoc(callee=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/1837076449.py', line=14, col=11), caller=Loc(file='<pipeline>', line=1, col=0))), Node(op='core.env', type=f64, args=(), regions=(), attrs=(('slot', (1, 0)),), loc=CallLoc(callee=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/1837076449.py', line=23, col=15), caller=Loc(file='<pipeline>', line=2, col=0)))), regions=(), attrs=(), loc=CallLoc(callee=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83191/1837076449.py', line=23, col=11), caller=Loc(file='<pipeline>', line=2, col=0))),), regions=(), attrs=(), loc=None),))

## What we can't do yet

- **Nothing calls `lower_handle` automatically** — dispatch (typeof the
  arguments, build the key, probe the cache, lower on miss) is ch09's hot
  path. Today we pass `arg_types` by hand.
- **Overloads** (`sqrt`, methods) await the Registry (step 8's surfaces);
  calls resolve only to casts and captured kernels.
- **Tuples, attributes, `if`/`for` statements** await their dialect packs.
- **Guards are still synthetic** — the base pack refuses globals outright,
  so there is nothing to guard yet; folded-global fates arrive with a
  policy, and their guards with them.

**Budget after step 6:** kernel 829 / 1150; satellites: combinators 163/250,
stdlib 83/1500 (the honesty-clause bucket, now separately counted).

**Next:** ch08 — one value, N parameters: bidirectional marshaling
(PackPlan + ResultPlan), leaves, and the `abi.slot` stage.